# Wake Word "Letícia" para Home Assistant (v7 — CUDA 12 Fixado)

**Correção principal vs v6:** PyTorch/torchaudio instalados com `--index-url https://download.pytorch.org/whl/cu121` para garantir alinhamento com `libcudart.so.12` do Colab, eliminando o `OSError: libcudart.so.13`.

## Instruções
1. **GPU T4 ativa** (Ambiente de execução > Alterar tipo > T4 GPU)
2. Execute **Etapa 1a** → Clique **Reiniciar sessão** quando solicitado
3. Execute **Etapa 1b em diante** (pode usar Executar tudo a partir daqui)
4. O modelo `.tflite` será baixado automaticamente ao final

> **Tempo total estimado:** ~2-3 horas (inclui download de ~17 GB de features)

---

## Etapa 0: Diagnóstico do Ambiente
Execute primeiro para confirmar a versão do CUDA disponível.

In [ ]:
# Diagnóstico do ambiente CUDA (execute antes de qualquer instalação)
print('=== DIAGNÓSTICO CUDA ===')
!nvidia-smi | head -10
print()
!nvcc --version 2>/dev/null || echo 'nvcc não encontrado'
print()
print('Bibliotecas CUDA disponíveis:')
!ldconfig -p 2>/dev/null | grep libcudart || find /usr/local/cuda* /usr/lib -name 'libcudart*' 2>/dev/null | head -5
print()
print('=== FIM DO DIAGNÓSTICO ===')

## Etapa 1a: Instalação de Dependências
⚠️ **Reiniciar sessão OBRIGATÓRIO após esta célula!**

**Correção v7:** PyTorch instalado via `--index-url https://download.pytorch.org/whl/cu121` + `--no-deps` no openwakeword para evitar upgrade automático para CUDA 13.

In [ ]:
import os, locale
locale.getpreferredencoding = lambda *a: 'UTF-8'

print('=' * 60)
print('  ETAPA 1a: Instalação de Dependências (CUDA 12 fixado)')
print('=' * 60)

# Clonar repositórios
if not os.path.exists('./piper-sample-generator'):
    !git clone -q https://github.com/dscripka/piper-sample-generator
    print('[OK] piper-sample-generator clonado')
else:
    print('[SKIP] piper-sample-generator já existe')

if not os.path.exists('./openwakeword'):
    !git clone -q https://github.com/dscripka/openWakeWord openwakeword
    print('[OK] openWakeWord clonado')
else:
    print('[SKIP] openWakeWord já existe')

# ============================================================
# FASE 1: Remover PyTorch pré-instalado do Colab
# O Colab vem com PyTorch que pode ser +cu130 (CUDA 13)
# Precisamos substituir por +cu121 (CUDA 12.1 - compatível com Colab)
# ============================================================
print('\nFase 1: Removendo PyTorch pré-instalado...')
!pip uninstall -y torch torchvision torchaudio torchmetrics 2>/dev/null || true
print('[OK] PyTorch removido')

# ============================================================
# FASE 2: Instalar PyTorch cu121 (CUDA 12.1)
# cu121 é forward-compatible com CUDA 12.4/12.5 do Colab
# Garante que libcudart.so.12 seja usado (não .so.13)
# ============================================================
print('\nFase 2: Instalando PyTorch cu121...')
!pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 \
    --index-url https://download.pytorch.org/whl/cu121
print('[OK] PyTorch 2.4.0 + cu121 instalado')

# ============================================================
# FASE 3: Dependências do openWakeWord (pinadas)
# --no-deps no openwakeword: evita que pip sobrescreva torch cu121 com cu13x
# ============================================================
print('\nFase 3: Dependências do openWakeWord...')
!pip install -q pathvalidate piper-tts piper-phonemize-cross webrtcvad
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0
!pip install -q speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0
!pip install -q acoustics==0.2.6 scipy datasets==2.14.6
!pip install -q -e ./openwakeword --no-deps
print('[OK] Dependências do openWakeWord instaladas')

# ============================================================
# FASE 4: ONNX e TensorFlow (exportação do modelo)
# tensorflow-cpu==2.15.0 + keras==2.15.0: versões estáveis para TFLite export
# onnx2tf: alternativa moderna ao onnx_tf (que conflita com TF 2.19)
# ============================================================
print('\nFase 4: ONNX e TensorFlow...')
!pip install -q onnx==1.14.1 onnxruntime==1.16.3
!pip install -q onnx_graphsurgeon sng4onnx
!pip install -q tensorflow-cpu==2.15.0 keras==2.15.0
!pip install -q onnx2tf
!pip install -q ai-edge-litert
print('[OK] ONNX e TensorFlow instalados')

# Instalar piper-sample-generator
!cd piper-sample-generator && pip install -q -r requirements.txt --no-deps

print()
print('=' * 60)
print('  ⚠️  REINICIE A SESSÃO AGORA!')
print('  Ambiente de execução > Reiniciar sessão')
print('  Depois execute a partir da Etapa 1b')
print('=' * 60)

## ⚠️ REINICIAR SESSÃO (OBRIGATÓRIO)

**Ambiente de execução → Reiniciar sessão**

Depois execute a partir da Etapa 1b.

---

## Etapa 1b: Verificação pós-reinício + Downloads

In [ ]:
import os, locale
import numpy as np
locale.getpreferredencoding = lambda *a: 'UTF-8'

print('=' * 60)
print('  ETAPA 1b: Verificação pós-reinício + Downloads')
print('=' * 60)

# Verificar numpy
_ = np.random.RandomState(42)
print(f'numpy {np.__version__}: OK')

# Verificar GPU e CUDA
import torch
print(f'PyTorch {torch.__version__}')
if '+cu121' in torch.__version__ or '+cu12' in torch.__version__:
    print('✅ CUDA 12.x confirmado — sem risco de libcudart.so.13')
elif '+cu13' in torch.__version__:
    print('❌ ATENÇÃO: PyTorch cu13 detectado — volte e execute Etapa 1a novamente!')
else:
    print(f'⚠️  Versão CUDA do torch: {torch.__version__}')

# Verificar se torchaudio importa sem erro
try:
    import torchaudio
    print(f'✅ torchaudio {torchaudio.__version__}: OK')
except OSError as e:
    print(f'❌ torchaudio erro: {e}')
    print('   Solução: volte e execute Etapa 1a novamente')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('⚠️  GPU não disponível — ative T4 em Ambiente de execução > Alterar tipo')

# Baixar modelos do openWakeWord
models_dir = 'openwakeword/openwakeword/resources/models'
os.makedirs(models_dir, exist_ok=True)
base_url = 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1'
for fname in ['embedding_model.onnx', 'embedding_model.tflite',
              'melspectrogram.onnx', 'melspectrogram.tflite']:
    fpath = os.path.join(models_dir, fname)
    if not os.path.exists(fpath):
        !wget -q '{base_url}/{fname}' -O {fpath}
        print(f'[OK] {fname}')
    else:
        print(f'[SKIP] {fname}')

# LibriTTS
libritts = 'piper-sample-generator/models/en_US-libritts_r-medium.pt'
if not os.path.exists(libritts):
    os.makedirs('piper-sample-generator/models', exist_ok=True)
    url = 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
    print('Baixando LibriTTS v2...')
    !wget -q -O {libritts} '{url}'
    print('[OK] LibriTTS v2')
else:
    print('[SKIP] LibriTTS')

# Vozes Piper pt_BR
os.makedirs('piper_voices_ptbr', exist_ok=True)
hf_base = 'https://huggingface.co/rhasspy/piper-voices/resolve/main'
voices = {
    'pt_BR-faber-medium': 'pt/pt_BR/faber/medium',
    'pt_BR-edresson-low': 'pt/pt_BR/edresson/low',
}
for name, path in voices.items():
    onnx_path = f'piper_voices_ptbr/{name}.onnx'
    if not os.path.exists(onnx_path):
        !wget -q -O {onnx_path} '{hf_base}/{path}/{name}.onnx'
        !wget -q -O {onnx_path}.json '{hf_base}/{path}/{name}.onnx.json'
        print(f'[OK] {name}')
    else:
        print(f'[SKIP] {name}')

print()
print('[OK] ETAPA 1b CONCLUÍDA!')

## Etapa 2: Testar Pronúncia
Ouça os áudios. Confirme que "Letícia" soa correto.

In [ ]:
import subprocess, os
from IPython.display import Audio, display

print('=' * 60)
print('  ETAPA 2: Testando pronúncia')
print('=' * 60)

target_word = 'letícia'
os.makedirs('test_audio', exist_ok=True)

for name, label in [('pt_BR-faber-medium','Faber'), ('pt_BR-edresson-low','Edresson')]:
    out = f'test_audio/test_{label.lower()}.wav'
    r = subprocess.run(
        ['piper', '--model', f'piper_voices_ptbr/{name}.onnx', '--output_file', out],
        input=target_word, capture_output=True, text=True
    )
    if os.path.exists(out):
        print(f'Voz {label} (pt_BR):')
        display(Audio(out, autoplay=False))
    else:
        print(f'[ERRO] {label}: {r.stderr[:200]}')

print('[OK] ETAPA 2 CONCLUÍDA!')

## Etapa 3: Download de Dados Auxiliares

| Dado | Tamanho | Tempo estimado |
|------|---------|----------------|
| MIT RIRs | ~30 MB | ~2 min |
| AudioSet | ~1 GB | ~5 min |
| FMA | ~300 MB | ~2 min |
| ACAV100M features | **17.3 GB** | ~10-20 min |
| Validation features | 185 MB | ~1 min |

**Tempo total: ~30-45 minutos**

In [ ]:
import os
import numpy as np
import scipy.io.wavfile as wav
import datasets
from tqdm import tqdm
from pathlib import Path

print('=' * 60)
print('  ETAPA 3: Baixando dados auxiliares')
print('=' * 60)

# 3a. MIT Room Impulse Responses
rir_dir = 'mit_rirs'
if not os.path.exists(rir_dir) or len([f for f in os.listdir(rir_dir) if f.endswith('.wav')]) == 0:
    print('\n[3a] MIT Room Impulse Responses...')
    os.makedirs(rir_dir, exist_ok=True)
    rir_dataset = datasets.load_dataset(
        'davidscripka/MIT_environmental_impulse_responses',
        split='train', streaming=True
    )
    count = 0
    for row in tqdm(rir_dataset, desc='MIT RIRs'):
        name = row['audio']['path'].split('/')[-1]
        audio_array = np.array(row['audio']['array'])
        wav.write(os.path.join(rir_dir, name), 16000,
                  (audio_array * 32767).astype(np.int16))
        count += 1
    print(f'[OK] MIT RIRs: {count} arquivos')
else:
    n = len([f for f in os.listdir(rir_dir) if f.endswith('.wav')])
    print(f'[SKIP] MIT RIRs ({n} arquivos)')

# 3b. AudioSet
as_dir = 'audioset_16k'
if not os.path.exists(as_dir) or len(os.listdir(as_dir)) == 0:
    print('\n[3b] AudioSet background noise...')
    os.makedirs('audioset', exist_ok=True)
    os.makedirs(as_dir, exist_ok=True)
    fname = 'bal_train09.tar'
    link = f'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/{fname}'
    if not os.path.exists(f'audioset/{fname}'):
        !wget -q --show-progress -O audioset/{fname} '{link}'
    !cd audioset && tar -xf {fname} 2>/dev/null || true
    flac_files = list(Path('audioset/audio').glob('**/*.flac')) if os.path.exists('audioset/audio') else []
    if len(flac_files) > 0:
        print(f'  Convertendo {len(flac_files)} FLAC → WAV 16kHz...')
        audioset_ds = datasets.Dataset.from_dict({'audio': [str(i) for i in flac_files]})
        audioset_ds = audioset_ds.cast_column('audio', datasets.Audio(sampling_rate=16000))
        for row in tqdm(audioset_ds, desc='AudioSet'):
            name = row['audio']['path'].split('/')[-1].replace('.flac', '.wav')
            audio_array = np.array(row['audio']['array'])
            wav.write(os.path.join(as_dir, name), 16000,
                      (audio_array * 32767).astype(np.int16))
    n = len(os.listdir(as_dir))
    print(f'[OK] AudioSet: {n} clips')
else:
    print(f'[SKIP] AudioSet ({len(os.listdir(as_dir))} clips)')

# 3c. FMA
fma_dir = 'fma'
if not os.path.exists(fma_dir) or len(os.listdir(fma_dir)) == 0:
    print('\n[3c] FMA música (1 hora)...')
    os.makedirs(fma_dir, exist_ok=True)
    fma_dataset = datasets.load_dataset('rudraml/fma', name='small', split='train', streaming=True)
    fma_iter = iter(fma_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000)))
    n_clips = 120  # 1 hora de clips de 30s
    count = 0
    for i in tqdm(range(n_clips), desc='FMA'):
        try:
            row = next(fma_iter)
            name = row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')
            audio_array = np.array(row['audio']['array'])
            wav.write(os.path.join(fma_dir, name), 16000,
                      (audio_array * 32767).astype(np.int16))
            count += 1
        except StopIteration:
            break
    print(f'[OK] FMA: {count} clips')
else:
    print(f'[SKIP] FMA ({len(os.listdir(fma_dir))} clips)')

# 3d. ACAV100M features (17.3 GB)
acav_file = 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
if not os.path.exists(acav_file):
    print('\n[3d] ACAV100M features (17.3 GB — ~10-20 min)...')
    url = 'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
    !wget -q --show-progress -c '{url}'
    if os.path.exists(acav_file):
        size_gb = os.path.getsize(acav_file) / (1024**3)
        print(f'[OK] ACAV100M: {size_gb:.1f} GB')
    else:
        print('[ERRO] ACAV100M download falhou!')
else:
    size_gb = os.path.getsize(acav_file) / (1024**3)
    print(f'[SKIP] ACAV100M ({size_gb:.1f} GB)')

# 3e. Validation features
val_file = 'validation_set_features.npy'
if not os.path.exists(val_file):
    print('\n[3e] Validation features (185 MB)...')
    url = 'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy'
    !wget -q --show-progress '{url}'
    print('[OK] Validation features')
else:
    size_mb = os.path.getsize(val_file) / (1024**2)
    print(f'[SKIP] Validation features ({size_mb:.0f} MB)')

# Resumo
print()
print('Resumo dos dados:')
for d, label in [('mit_rirs','RIRs'), ('audioset_16k','AudioSet'), ('fma','FMA')]:
    if os.path.exists(d):
        n = len([f for f in os.listdir(d) if f.endswith('.wav')])
        print(f'  {label}: {n} arquivos WAV')
    else:
        print(f'  ❌ {label}: FALTANDO')
for f, label in [(acav_file,'ACAV100M'), (val_file,'Validation')]:
    if os.path.exists(f):
        s = os.path.getsize(f) / (1024**2)
        unit = 'GB' if s > 1024 else 'MB'
        val = s/1024 if s > 1024 else s
        print(f'  {label}: {val:.1f} {unit}')
    else:
        print(f'  ❌ {label}: FALTANDO')

print()
print('[OK] ETAPA 3 CONCLUÍDA!')

## Etapa 4: Gerar Clips com Piper pt_BR

| Tipo | Quantidade |
|------|----------|
| Positivos treino | 1.500 |
| Positivos validação | 500 |
| Negativos treino | 1.500 |
| Negativos validação | 500 |

**Tempo estimado: ~15-30 min (paralelo com 4 workers)**

In [ ]:
import os, subprocess, uuid, random, time
from concurrent.futures import ThreadPoolExecutor, as_completed

print('=' * 60)
print('  ETAPA 4: Gerando clips com Piper pt_BR')
print('=' * 60)

target_word = 'letícia'
model_name  = 'leticia'
n_positive  = 1500
n_val       = 500

ptbr_voices = [
    'piper_voices_ptbr/pt_BR-faber-medium.onnx',
    'piper_voices_ptbr/pt_BR-edresson-low.onnx',
]
length_scales = [0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25]
noise_scales  = [0.5, 0.6, 0.667, 0.7, 0.8, 0.9, 0.98]
noise_ws      = [0.5, 0.6, 0.7, 0.8, 0.9, 0.98]

negative_words = [
    'patrícia', 'notícia', 'delícia', 'justiça', 'preguiça',
    'milícia', 'malícia', 'polícia', 'carência', 'urgência',
    'letivo', 'letrada', 'legítima', 'legião', 'elétrica',
    'lícia', 'alícia', 'felícia', 'luciana', 'larissa',
    'olá', 'bom dia', 'boa noite', 'obrigado', 'por favor',
    'ligar', 'desligar', 'acender', 'apagar', 'aumentar',
    'diminuir', 'temperatura', 'música', 'que horas são',
    'televisão', 'computador', 'celular', 'internet', 'cozinha',
]

base = f'./my_custom_model/{model_name}'
dirs = {
    'positive_train': f'{base}/positive_train',
    'positive_test':  f'{base}/positive_test',
    'negative_train': f'{base}/negative_train',
    'negative_test':  f'{base}/negative_test',
}
for d in dirs.values():
    os.makedirs(d, exist_ok=True)

def gen_one_clip(args):
    word, voice, out_path = args
    try:
        subprocess.run(
            ['piper', '--model', voice, '--output_file', out_path,
             '--length-scale', str(random.choice(length_scales)),
             '--noise-scale', str(random.choice(noise_scales)),
             '--noise-w', str(random.choice(noise_ws))],
            input=word, capture_output=True, text=True, timeout=30
        )
        return os.path.exists(out_path)
    except Exception:
        return False

def gen_clips_parallel(text, out_dir, n, label, workers=4):
    existing = len([f for f in os.listdir(out_dir) if f.endswith('.wav')])
    if existing >= int(n * 0.95):
        print(f'  [SKIP] {label}: {existing} clips já existem')
        return
    needed = n - existing
    texts = [text] if isinstance(text, str) else text
    tasks = []
    for i in range(needed):
        word = random.choice(texts) if isinstance(texts, list) else texts
        voice = random.choice(ptbr_voices)
        out = os.path.join(out_dir, f'{uuid.uuid4().hex}.wav')
        tasks.append((word, voice, out))
    count = 0
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(gen_one_clip, t): t for t in tasks}
        for future in as_completed(futures):
            if future.result():
                count += 1
            if count % 200 == 0 and count > 0:
                elapsed = time.time() - t0
                rate = count / elapsed
                remaining = (needed - count) / rate if rate > 0 else 0
                print(f'  {label}: {count}/{needed} ({rate:.1f} clips/s, ~{remaining/60:.0f} min)')
    total = len([f for f in os.listdir(out_dir) if f.endswith('.wav')])
    print(f'  [OK] {label}: {total} clips ({(time.time()-t0)/60:.1f} min)')

N_WORKERS = 4
print(f'Usando {N_WORKERS} workers paralelos\n')
gen_clips_parallel(target_word,    dirs['positive_train'], n_positive, 'Positivos treino', N_WORKERS)
gen_clips_parallel(target_word,    dirs['positive_test'],  n_val,      'Positivos val',    N_WORKERS)
gen_clips_parallel(negative_words, dirs['negative_train'], n_positive, 'Negativos treino', N_WORKERS)
gen_clips_parallel(negative_words, dirs['negative_test'],  n_val,      'Negativos val',    N_WORKERS)

print('\nResumo:')
for label, d in dirs.items():
    n = len([f for f in os.listdir(d) if f.endswith('.wav')])
    print(f'  {label}: {n} clips')
print('\n[OK] ETAPA 4 CONCLUÍDA!')

## Etapa 5: Augmentação + Extração de Features

**Correção v7:** torchaudio importa corretamente com cu121 — sem OSError.

**Tempo estimado: ~15-30 min**

In [ ]:
import os, sys, yaml
import numpy as np

print('=' * 60)
print('  ETAPA 5: Augmentação e extração de features')
print('=' * 60)

# Verificar torchaudio antes de prosseguir
try:
    import torchaudio
    print(f'✅ torchaudio {torchaudio.__version__}: OK')
except OSError as e:
    print(f'❌ torchaudio ERRO: {e}')
    print('Solução: reinicie e execute Etapa 1a novamente com install cu121')
    raise

model_name = 'leticia'

# RIR path
if os.path.exists('mit_rirs') and len([f for f in os.listdir('mit_rirs') if f.endswith('.wav')]) > 0:
    rir_path = os.path.abspath('mit_rirs')
else:
    rir_path = os.path.abspath('piper-sample-generator/impulses')
print(f'RIR: {rir_path} ({len(os.listdir(rir_path))} arquivos)')

# Verificar AudioSet — aceitar diretório mesmo vazio (FMA é suficiente)
audioset_path = os.path.abspath('./audioset_16k')
fma_path = os.path.abspath('./fma')
n_audioset = len([f for f in os.listdir(audioset_path) if f.endswith('.wav')]) if os.path.exists(audioset_path) else 0
n_fma = len([f for f in os.listdir(fma_path) if f.endswith('.wav')]) if os.path.exists(fma_path) else 0
print(f'AudioSet: {n_audioset} clips | FMA: {n_fma} clips')

# Garantir que background_paths só aponta para dirs com arquivos
background_paths = []
if n_audioset > 0:
    background_paths.append(audioset_path)
if n_fma > 0:
    background_paths.append(fma_path)
if not background_paths:
    print('⚠️  Sem dados de background! Usando RIRs como fallback')
    background_paths = [rir_path]

config = {
    'model_name': model_name,
    'target_phrase': ['letícia'],
    'custom_negative_phrases': [
        'patrícia', 'notícia', 'delícia', 'justiça',
        'milícia', 'malícia', 'polícia', 'alícia', 'felícia'
    ],
    'n_samples': 1500,
    'n_samples_val': 500,
    'tts_batch_size': 50,
    'augmentation_batch_size': 16,
    'piper_sample_generator_path': os.path.abspath('./piper-sample-generator'),
    'output_dir': os.path.abspath('./my_custom_model'),
    'rir_paths': [rir_path],
    'background_paths': background_paths,
    'background_paths_duplication_rate': [1] * len(background_paths),
    'augmentation_rounds': 1,
    'false_positive_validation_data_path': os.path.abspath('./validation_set_features.npy'),
    'feature_data_files': {
        'ACAV100M_sample': os.path.abspath('./openwakeword_features_ACAV100M_2000_hrs_16bit.npy')
    },
    'batch_n_per_class': {
        'ACAV100M_sample': 1024,
        'adversarial_negative': 50,
        'positive': 50
    },
    'model_type': 'dnn',
    'layer_size': 32,
    'steps': 50000,
    'max_negative_weight': 1500,
    'target_false_positives_per_hour': 0.2,
}

with open('my_model.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print('Config salva: my_model.yaml')

# Augmentação
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

# Verificar saída
feature_dir = f'my_custom_model/{model_name}'
expected_files = ['positive_features_train.npy', 'negative_features_train.npy',
                  'positive_features_test.npy', 'negative_features_test.npy']
print()
print('Features gerados:')
all_ok = True
for f in expected_files:
    fp = os.path.join(feature_dir, f)
    if os.path.exists(fp):
        shape = np.load(fp, mmap_mode='r').shape
        print(f'  ✅ {f}: {shape}')
    else:
        print(f'  ❌ {f}: NÃO ENCONTRADO')
        all_ok = False

if all_ok:
    print('\n[OK] ETAPA 5 CONCLUÍDA!')
else:
    print('\n[ERRO] Alguns features não foram gerados. Verifique os logs acima.')

## Etapa 6: Treinar o Modelo

**Tempo estimado: ~30-60 min com GPU T4**

In [ ]:
import sys

print('=' * 60)
print('  ETAPA 6: Treinando modelo')
print('=' * 60)

!{sys.executable} openwakeword/openwakeword/train.py \
    --training_config my_model.yaml \
    --train_model

print()
print('[OK] ETAPA 6 CONCLUÍDA!')

## Etapa 7: Converter para TFLite e Baixar

**Correção v7:** usa `tensorflow-cpu==2.15.0` + `onnx2tf` como fallback — mais estável que `onnx_tf` no Colab.

In [ ]:
import os, glob, shutil
from google.colab import files

print('=' * 60)
print('  ETAPA 7: Modelo final')
print('=' * 60)

model_name  = 'leticia'
onnx_path   = f'my_custom_model/{model_name}.onnx'
tflite_path = f'my_custom_model/{model_name}.tflite'

# Verificar se train.py já gerou TFLite
if os.path.exists(onnx_path) and not os.path.exists(tflite_path):
    print('TFLite não gerado. Convertendo via onnx2tf...')
    tf_out = f'my_custom_model/tf_model_{model_name}'
    !onnx2tf -i {onnx_path} -o {tf_out} -oiqt 2>&1 | tail -20
    tflite_files = glob.glob(f'{tf_out}/**/*.tflite', recursive=True)
    if tflite_files:
        # Preferir o modelo sem quantização para maior acurácia
        float_models = [f for f in tflite_files if 'float32' in f or 'float16' in f]
        chosen = float_models[0] if float_models else tflite_files[0]
        shutil.copy2(chosen, tflite_path)
        print(f'[OK] TFLite gerado: {os.path.basename(chosen)}')
    else:
        print('[ERRO] Conversão TFLite falhou!')

# Relatório
print()
for path, label in [(onnx_path, 'ONNX'), (tflite_path, 'TFLite')]:
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f'  ✅ {label}: {path} ({size:.1f} KB)')
    else:
        print(f'  ❌ {label}: não encontrado')

# Download
print()
for path in [tflite_path, onnx_path]:
    if os.path.exists(path):
        print(f'Baixando {os.path.basename(path)}...')
        files.download(path)

print()
print('=' * 60)
print('  🎉 CONCLUÍDO!')
print()
print('  Deploy no Home Assistant Yellow:')
print('  1. Copie leticia.tflite para /share/openwakeword/')
print('  2. Reinicie o add-on openWakeWord')
print('  3. Configure: Assistants > Wake Word > leticia')
print('  4. Teste: "Letícia, que horas são?"')
print('=' * 60)